# 多端口校准

本示例演示了如何使用 `MultiportSOLT` 校准类来校准具有三个或更多端口的多端口散射参数测量。

## 设置

In [ ]:
import numpy as np

import skrf
from skrf.calibration import SOLT, MultiportCal, MultiportSOLT, UnknownThru, terminate_nport
from skrf.frequency import Frequency
from skrf.media import Coaxial
from skrf.network import connect, twoport_to_nport

skrf.stylely()

## 创建合成数据首先，我们创建一个合成误差网络。

In [ ]:
freq = Frequency(1,100,100, unit='GHz')
nports = 3
# 1.0 mm coaxial media for calibration error boxes
coax = Coaxial(freq, z0_port=50, Dint=0.44e-3, Dout=1.0e-3, sigma=1e8)

# Create random error network
# First 'nports' ports will connect to an ideal VNA
# and the last 'nports' ports will connect to the DUT.
Z = coax.random(n_ports = 2*nports, name = 'Z')

# Zero leakage terms in the error network.
port_type = lambda n: 'VNA' if n < nports else 'DUT'  # noqa: E731
port_number = lambda n: n if n < nports else n - nports  # noqa: E731
for i in range(2*nports):
    for j in range(i+1, 2*nports):
        # No connection between different VNA/DUT ports.
        # No connection between VNA and DUT ports with different number.
        if port_type(i) == port_type(j) or port_number(i) != port_number(j):
            Z.s[:,i,j] = 0
            Z.s[:,j,i] = 0

# VNA switch terms. This is the termination impedance of each port
# when the source is not connected to that port.
gammas = []
for i in range(nports):
    gammas.append(coax.random(n_ports=1, name=f'gamma_{i}'))

该函数用于通过误差网络来测量网络。只需将误差网络连接到待测网络，并添加终端阻抗即可。

In [ ]:
def measure(ntwk, Z, gammas, nports):
    out = terminate_nport(connect(Z, nports, ntwk, 0, num=nports), gammas)
    out.name = ntwk.name
    return out

接下来，创建校准标准。这些标准应该是多端口网络。在实际的 VNA 测量中，可能需要手动将二端口标准的测量结果组合成 N 端口的标准。所需的标准包括：在每个端口上进行开路、短路和匹配测量，以及从第一个端口到所有其他端口的直通测量。

In [ ]:
o = coax.open(nports=nports, name='open')
s = coax.short(nports=nports, name='short')
m = coax.match(nports=nports, name='load')

thru = coax.thru(name='thru')

ideals = []
# nports-1 thrus from port 0 to all other ports.
for i in range(1, nports):
    thru_i = twoport_to_nport(thru, 0, i, nports)
    ideals.append(thru_i)

ideals.extend([o,s,m])
measured = [measure(k, Z, gammas, nports) for k in ideals]

用于测试校准的设备。一个简单的三端口连接器。由于误差网络是随机生成的，并且具有非常高的反射，因此绘制的未校准网络的散射参数无法识别。

In [ ]:
dut = coax.tee(name='dut')
dut_meas = measure(dut, Z, gammas, nports)
dut_meas.plot_s_db()

## 校准

创建 `MultiportSOLT` 校准类。第一个参数是该类用于校准多端口设备的双端口校准方法。在这种情况下，我们使用简单的 SOLT 校准。其他可能的选项包括 `UnknownThru` 和 `LRRM`。请注意，不同的校准算法可能需要以特定顺序提供校准标准，并且可能需要其他输入，例如 `switch_terms`。请参考双端口校准文档，了解所需的输入。

In [ ]:
cal = MultiportSOLT(method=SOLT, measured=measured, ideals=ideals)

校准待测器件 (DUT)，并绘制校准后的散射参数。

In [ ]:
dut_cal = cal.apply_cal(dut_meas)
dut_cal.plot_s_db()

经过校正后的DUT（被测器件）散射参数与原始散射参数之间的差异应该很小。

In [ ]:
np.max(np.abs(dut_cal.s - dut.s))

## 多端口校准上述 `MultiportSOLT` 类只能用于 SOLT 风格的二端口校准，该校准只需要在两个端口之间使用一个透射标准。多端口 TRL 校准需要在端口之间使用直通标准和一个或多个线路，这与上述类的接口不匹配。此外，如果出于某种原因，需要为不同的端口对使用不同的二端口校准方法，则 `MultiportSOLT` 类无法做到这一点。对于这些情况，可以使用较低级别的 `MultiportCal`。此校准需要一个包含端口对的字典作为输入。'method' 键是二端口校准类，'measured' 是测量的网络。它们可以是二端口或 N 端口。其他键将传递给二端口校准例程。在这种情况下，我们使用 `UnknownThru` 类，该类需要额外的 `switch_term` 参数。请注意，二端口开关项的顺序与多端口开关项的顺序相反。二端口开关项的列出顺序是源连接的位置，但在多端口开关项中，此顺序未指定，并且它们是按照它们终止的端口的顺序列出的。同样重要的是，按照二端口校准所需的正确顺序列出理想值和测量的网络。`UnknownThru` 假定直通标准应列在最后，并且其他标准的顺序无关紧要。

In [ ]:
ideals_01 = ideals[2:] + [ideals[0]]
ideals_02 = ideals[2:] + [ideals[1]]
meas_01 = measured[2:] + [measured[0]]
meas_02 = measured[2:] + [measured[1]]

cal_dict = {(0, 1): {'method': UnknownThru, 'ideals': ideals_01,
                     'measured': meas_01, 'switch_terms':[gammas[1], gammas[0]]},
            (0, 2): {'method': UnknownThru, 'ideals': ideals_02,
                     'measured': meas_02, 'switch_terms':[gammas[2], gammas[0]]}}

cal2 = MultiportCal(cal_dict)

dut_cal2 = cal2.apply_cal(dut_meas)
dut_cal2.plot_s_db()